## Coordination Score & Tiered Enforcement

In [ ]:
# Finding bad videos is useless without a plan. 
# YouTube can't manually review everything. 
# We build a composite score from all signals, then assign each video to an enforcement tier 
# -- just like real Trust & Safety teams do

#The Coordination Score Formula:
#• +2 points -- Flagged by Isolation Forest (basic anomaly)
# • +3 points -- Part of a bot network (5+ videos) -- STRONGEST signal
# • +1.5 points -- Spiked at suspicious hour (bot schedule)
# • +2 points -- Co-spiked with other videos (temporal coordination)
# • +1.5 points -- Low sustainability + high spike (classic bot signature)

#Enforcement Tiers:
# • Tier 1 (Score 6+): Confirmed bot network -> Auto-action (remove fake engagement, demonetize)
# • Tier 2 (Score 5-6): Likely bot family -> Human investigator reviews the network
# • Tier 3 (Score 4-5): Suspicious solo -> Watchlist, monitor for repeat behavior
# • Tier 4 (Score 2-4): Low confidence solo -> Re-check next cycle, possible false positive
# • Clean (Score 0-1): No action needed

In [4]:
import pandas as pd
import numpy as np

In [7]:
video_features = pd.read_csv("data/processed/engineered_video_features3.csv")

In [ ]:
# ============================================================
#  COORDINATED BOT FARM SCORE — Combine all signals
# ============================================================

In [11]:
def calculate_coordination_score_v2(row):
    score = 0
    
    # Base: Isolation Forest flag
    if row['is_suspicious_v2']:
        score += 2
    
    # Part of a confirmed bot network (multi-video coordination)
    if row['in_bot_network'] and row['bot_network_size'] >= 3:
        score += 3  # Higher weight — this is the strongest signal
    
    # Spiked at a suspicious hour (bot schedule)
    if row.get('is_suspicious_hour', False):
        score += 1.5
    
    # Co-spiked with other videos
    if row.get('is_cospike', False):
        score += 2
    
    # Classic bot signature: low sustainability + high spike
    if row['sustainability'] < 0.1 and row['like_spike_ratio'] > 20:
        score += 1.5
    
    # Solo suspicious but no network — still worth reviewing
    if row['is_suspicious_v2'] and not row['in_bot_network'] and row['like_spike_ratio'] > 15:
        score += 1
    
    return score

video_features['coordination_score'] = video_features.apply(calculate_coordination_score_v2, axis=1)

# Updated tiers
def assign_tier_v3(row):
    if not row['is_suspicious_v2'] and row['coordination_score'] < 2:
        return "clean"
    elif row['in_bot_network'] and row['bot_network_size'] >= 5:
        return "tier1_confirmed_bot_network"  # Auto-action, high confidence
    elif row['in_bot_network'] and row['bot_network_size'] >= 3:
        return "tier2_likely_bot_family"      # Human review + network analysis
    elif row['coordination_score'] >= 4:
        return "tier3_suspicious_solo"        # Watchlist
    elif row['is_suspicious_v2']:
        return "tier4_low_confidence_solo"    # Re-check next cycle
    else:
        return "clean"

video_features['enforcement_tier_v3'] = video_features.apply(assign_tier_v3, axis=1)

print("\n=== NEW TIER DISTRIBUTION (Bot Network Aware) ===")
print(video_features['enforcement_tier_v3'].value_counts())


=== NEW TIER DISTRIBUTION (Bot Network Aware) ===
enforcement_tier_v3
clean                          9496
tier1_confirmed_bot_network     262
tier4_low_confidence_solo       178
tier3_suspicious_solo            42
tier2_likely_bot_family          21
Name: count, dtype: int64


In [12]:
def assign_tier_v2(row):
    if not row['is_suspicious_v2'] and row['coordination_score'] < 2:
        return "clean"
    elif row['coordination_score'] >= 6:
        return "tier1_coordinated_bot_farm"  # Auto-action, high confidence
    elif row['coordination_score'] >= 4:
        return "tier2_likely_coordinated"      # Human review + network analysis
    elif row['coordination_score'] >= 2:
        return "tier3_suspicious_solo"         # Watchlist, re-check next cycle
    else:
        return "tier4_low_confidence"

video_features['enforcement_tier_v2'] = video_features.apply(assign_tier_v2, axis=1)

print("\n=== NEW TIER DISTRIBUTION (with Coordination Detection) ===")
print(video_features['enforcement_tier_v2'].value_counts())


=== NEW TIER DISTRIBUTION (with Coordination Detection) ===
enforcement_tier_v2
clean                         9219
tier3_suspicious_solo          455
tier2_likely_coordinated       294
tier1_coordinated_bot_farm      31
Name: count, dtype: int64


In [13]:
video_features.to_csv("data/processed/engineered_video_features_final.csv", index=False)